In [1]:
import pandas as pd

# Membaca data utama
df = pd.read_csv("3Polutan_Dukun_Terkini.csv")

# Pisahkan masing-masing polutan
df_NO2 = df[['date', 'NO2']]
df_CO  = df[['date', 'CO']]
df_SO2 = df[['date', 'SO2']]

# Simpan
df_NO2.to_csv("NO2_Dukun_Gresik.csv", index=False)
df_CO.to_csv("CO_Dukun_Gresik.csv", index=False)
df_SO2.to_csv("SO2_Dukun_Gresik.csv", index=False)

print("Berhasil memisahkan data!")

Berhasil memisahkan data!


## Preprocesing

In [2]:
import pandas as pd
import numpy as np


# ============================================================
# FUNGSI PREPROCESSING LINEAR
# ============================================================

def preprocessing_linear(
    file_input,
    pollutant,
    file_output
):

    # --------------------------------------------------------
    # 1. Membaca data
    # --------------------------------------------------------

    df = pd.read_csv(file_input)

    # Ubah date menjadi datetime
    df['date'] = pd.to_datetime(df['date'])

    # Urutkan berdasarkan tanggal
    df = df.sort_values('date').reset_index(drop=True)

    # Pastikan nilai polutan berupa numerik
    df[pollutant] = pd.to_numeric(
        df[pollutant],
        errors='coerce'
    )


    # --------------------------------------------------------
    # 2. Deteksi Outlier dengan IQR
    # --------------------------------------------------------

    Q1 = df[pollutant].quantile(0.25)

    Q3 = df[pollutant].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - (1.5 * IQR)

    batas_atas = Q3 + (1.5 * IQR)


    # Identifikasi outlier
    outlier_mask = (
        (df[pollutant] < batas_bawah) |
        (df[pollutant] > batas_atas)
    )

    jumlah_outlier = outlier_mask.sum()


    # --------------------------------------------------------
    # 3. Outlier menjadi NaN
    # --------------------------------------------------------

    df.loc[
        outlier_mask,
        pollutant
    ] = np.nan


    # --------------------------------------------------------
    # 4. Lengkapi tanggal
    # --------------------------------------------------------

    df = df.set_index('date')

    rentang_tanggal = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq='D'
    )

    df = df.reindex(rentang_tanggal)


    # --------------------------------------------------------
    # 5. INTERPOLASI LINEAR
    # --------------------------------------------------------

    df[pollutant] = df[pollutant].interpolate(
        method='linear'
    )


    # --------------------------------------------------------
    # 6. Mengatasi NaN awal / akhir
    # --------------------------------------------------------

    df[pollutant] = (
        df[pollutant]
        .bfill()
        .ffill()
    )


    # --------------------------------------------------------
    # 7. Kembalikan date menjadi kolom
    # --------------------------------------------------------

    df.index.name = 'date'

    df = df.reset_index()


    # --------------------------------------------------------
    # 8. Simpan
    # --------------------------------------------------------

    df.to_csv(
        file_output,
        index=False
    )


    # --------------------------------------------------------
    # 9. Informasi hasil
    # --------------------------------------------------------

    print("=" * 50)
    print(f"POLUTAN : {pollutant}")
    print(f"Outlier ditemukan : {jumlah_outlier}")
    print(
        f"Missing value akhir : "
        f"{df[pollutant].isna().sum()}"
    )
    print(f"File : {file_output}")
    print()


# ============================================================
# PROSES NO2
# ============================================================

preprocessing_linear(
    "NO2_Dukun_Gresik.csv",
    "NO2",
    "NO2_Dukun_Gresik_Linear.csv"
)


# ============================================================
# PROSES CO
# ============================================================

preprocessing_linear(
    "CO_Dukun_Gresik.csv",
    "CO",
    "CO_Dukun_Gresik_Linear.csv"
)


# ============================================================
# PROSES SO2
# ============================================================

preprocessing_linear(
    "SO2_Dukun_Gresik.csv",
    "SO2",
    "SO2_Dukun_Gresik_Linear.csv"
)

POLUTAN : NO2
Outlier ditemukan : 22
Missing value akhir : 0
File : NO2_Dukun_Gresik_Linear.csv

POLUTAN : CO
Outlier ditemukan : 4
Missing value akhir : 0
File : CO_Dukun_Gresik_Linear.csv

POLUTAN : SO2
Outlier ditemukan : 15
Missing value akhir : 0
File : SO2_Dukun_Gresik_Linear.csv



In [3]:
import pandas as pd


# ============================================================
# MEMBACA HASIL PREPROCESSING
# ============================================================

df_no2 = pd.read_csv(
    "NO2_Dukun_Gresik_Linear.csv"
)

df_co = pd.read_csv(
    "CO_Dukun_Gresik_Linear.csv"
)

df_so2 = pd.read_csv(
    "SO2_Dukun_Gresik_Linear.csv"
)


# ============================================================
# FORMAT DATE
# ============================================================

df_no2['date'] = pd.to_datetime(df_no2['date'])
df_co['date']  = pd.to_datetime(df_co['date'])
df_so2['date'] = pd.to_datetime(df_so2['date'])


# ============================================================
# GABUNGKAN
# ============================================================

df_linear = df_no2[['date', 'NO2']].merge(
    df_co[['date', 'CO']],
    on='date',
    how='outer'
)

df_linear = df_linear.merge(
    df_so2[['date', 'SO2']],
    on='date',
    how='outer'
)


# Urutkan tanggal
df_linear = df_linear.sort_values(
    'date'
).reset_index(drop=True)


# ============================================================
# CEK
# ============================================================

print(df_linear.head())

print("\nJumlah missing:")
print(df_linear.isna().sum())


# ============================================================
# SIMPAN
# ============================================================

df_linear.to_csv(
    "Polutan_Dukun_Gresik_Linear.csv",
    index=False
)

print(
    "\nBerhasil membuat "
    "Polutan_Dukun_Gresik_Linear.csv"
)

        date       NO2        CO       SO2
0 2025-08-31  0.000029  0.025446 -0.000006
1 2025-09-01  0.000040  0.027305 -0.000235
2 2025-09-02  0.000035  0.024726 -0.000133
3 2025-09-03  0.000031  0.026264 -0.000032
4 2025-09-04  0.000021  0.027802  0.000078

Jumlah missing:
date    0
NO2     0
CO      0
SO2     0
dtype: int64

Berhasil membuat Polutan_Dukun_Gresik_Linear.csv


polynomial

In [4]:
import pandas as pd
import numpy as np


# ============================================================
# FUNGSI PREPROCESSING POLYNOMIAL
# ============================================================

def preprocessing_polynomial(
    file_input,
    pollutant,
    file_output,
    order=2
):

    # --------------------------------------------------------
    # 1. Membaca data
    # --------------------------------------------------------

    df = pd.read_csv(file_input)

    # Date
    df['date'] = pd.to_datetime(df['date'])

    # Urutkan tanggal
    df = df.sort_values('date').reset_index(drop=True)

    # Pastikan numerik
    df[pollutant] = pd.to_numeric(
        df[pollutant],
        errors='coerce'
    )


    # --------------------------------------------------------
    # 2. Deteksi Outlier IQR
    # --------------------------------------------------------

    Q1 = df[pollutant].quantile(0.25)

    Q3 = df[pollutant].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - (1.5 * IQR)

    batas_atas = Q3 + (1.5 * IQR)


    # Mask outlier
    outlier_mask = (
        (df[pollutant] < batas_bawah) |
        (df[pollutant] > batas_atas)
    )

    jumlah_outlier = outlier_mask.sum()


    # --------------------------------------------------------
    # 3. Outlier → NaN
    # --------------------------------------------------------

    df.loc[
        outlier_mask,
        pollutant
    ] = np.nan


    # --------------------------------------------------------
    # 4. Lengkapi tanggal
    # --------------------------------------------------------

    df = df.set_index('date')

    rentang_tanggal = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq='D'
    )

    df = df.reindex(rentang_tanggal)


    # --------------------------------------------------------
    # 5. INTERPOLASI POLYNOMIAL
    # --------------------------------------------------------

    df[pollutant] = df[pollutant].interpolate(
        method='polynomial',
        order=order
    )


    # --------------------------------------------------------
    # 6. NaN awal / akhir
    # --------------------------------------------------------

    df[pollutant] = (
        df[pollutant]
        .bfill()
        .ffill()
    )


    # --------------------------------------------------------
    # 7. Kembalikan date
    # --------------------------------------------------------

    df.index.name = 'date'

    df = df.reset_index()


    # --------------------------------------------------------
    # 8. Simpan
    # --------------------------------------------------------

    df.to_csv(
        file_output,
        index=False
    )


    # --------------------------------------------------------
    # 9. Informasi
    # --------------------------------------------------------

    print("=" * 50)
    print(f"POLUTAN : {pollutant}")
    print(f"Outlier ditemukan : {jumlah_outlier}")
    print(
        f"Missing value akhir : "
        f"{df[pollutant].isna().sum()}"
    )
    print(f"File : {file_output}")
    print()


# ============================================================
# NO2
# ============================================================

preprocessing_polynomial(
    "NO2_Dukun_Gresik.csv",
    "NO2",
    "NO2_Dukun_Gresik_Polynomial.csv",
    order=2
)


# ============================================================
# CO
# ============================================================

preprocessing_polynomial(
    "CO_Dukun_Gresik.csv",
    "CO",
    "CO_Dukun_Gresik_Polynomial.csv",
    order=2
)


# ============================================================
# SO2
# ============================================================

preprocessing_polynomial(
    "SO2_Dukun_Gresik.csv",
    "SO2",
    "SO2_Dukun_Gresik_Polynomial.csv",
    order=2
)

POLUTAN : NO2
Outlier ditemukan : 22
Missing value akhir : 0
File : NO2_Dukun_Gresik_Polynomial.csv

POLUTAN : CO
Outlier ditemukan : 4
Missing value akhir : 0
File : CO_Dukun_Gresik_Polynomial.csv

POLUTAN : SO2
Outlier ditemukan : 15
Missing value akhir : 0
File : SO2_Dukun_Gresik_Polynomial.csv



In [5]:
import pandas as pd


# ============================================================
# MEMBACA HASIL
# ============================================================

df_no2 = pd.read_csv(
    "NO2_Dukun_Gresik_Polynomial.csv"
)

df_co = pd.read_csv(
    "CO_Dukun_Gresik_Polynomial.csv"
)

df_so2 = pd.read_csv(
    "SO2_Dukun_Gresik_Polynomial.csv"
)


# ============================================================
# FORMAT DATE
# ============================================================

df_no2['date'] = pd.to_datetime(df_no2['date'])
df_co['date']  = pd.to_datetime(df_co['date'])
df_so2['date'] = pd.to_datetime(df_so2['date'])


# ============================================================
# GABUNGKAN
# ============================================================

df_polynomial = df_no2[['date', 'NO2']].merge(
    df_co[['date', 'CO']],
    on='date',
    how='outer'
)

df_polynomial = df_polynomial.merge(
    df_so2[['date', 'SO2']],
    on='date',
    how='outer'
)


# Urutkan
df_polynomial = df_polynomial.sort_values(
    'date'
).reset_index(drop=True)


# ============================================================
# CEK
# ============================================================

print(df_polynomial.head())

print("\nJumlah missing:")
print(df_polynomial.isna().sum())


# ============================================================
# SIMPAN
# ============================================================

df_polynomial.to_csv(
    "Polutan_Dukun_Gresik_Polynomial.csv",
    index=False
)

print(
    "\nBerhasil membuat "
    "Polutan_Dukun_Gresik_Polynomial.csv"
)

        date       NO2        CO       SO2
0 2025-08-31  0.000029  0.025446 -0.000006
1 2025-09-01  0.000040  0.027305 -0.000235
2 2025-09-02  0.000035  0.024726 -0.000133
3 2025-09-03  0.000031  0.024734 -0.000032
4 2025-09-04  0.000021  0.027802  0.000078

Jumlah missing:
date    0
NO2     0
CO      0
SO2     0
dtype: int64

Berhasil membuat Polutan_Dukun_Gresik_Polynomial.csv


tsfel

In [6]:
!pip install tsfel

In [7]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat 1 file CSV utama yang berisi semua polutan ----------
df = pd.read_csv('Polutan_Dukun_Gresik_Linear.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar 68 Fitur TSFEL ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")

# ---------- 3. Fungsi ekstraksi dan penyeragaman output  ----------
def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Dictionary untuk menampung seluruh hasil ekstraksi
combined_row = {}

# ---------- 4. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    print(f"\n--- Memproses polutan: {pollutant} ---")

    df_poly = df[['date', pollutant]].copy()
    df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors='coerce')

    # Handling outlier dengan IQR (sebagai safeguard tambahan)
    Q1 = df_poly[pollutant].quantile(0.25)
    Q3 = df_poly[pollutant].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[pollutant].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 5. Simpan ke DataFrame final ----------
extracted_features_final = pd.DataFrame([combined_row])
print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = 'Dukun_gresik_linear.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

Jumlah fitur per polutan: 68
Total target fitur keseluruhan: 204

--- Memproses polutan: NO2 ---

--- Memproses polutan: SO2 ---



--- Memproses polutan: CO ---

Berhasil! Total kolom akhir yang dihasilkan: 204
File berhasil disimpan sebagai: Dukun_gresik_linear.csv


In [8]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat 1 file CSV utama yang berisi semua polutan ----------
df = pd.read_csv('Polutan_Dukun_Gresik_Polynomial.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar 68 Fitur TSFEL ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")

# ---------- 3. Fungsi ekstraksi dan penyeragaman output  ----------
def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Dictionary untuk menampung seluruh hasil ekstraksi
combined_row = {}

# ---------- 4. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    print(f"\n--- Memproses polutan: {pollutant} ---")

    df_poly = df[['date', pollutant]].copy()
    df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors='coerce')

    # Handling outlier dengan IQR (sebagai safeguard tambahan)
    Q1 = df_poly[pollutant].quantile(0.25)
    Q3 = df_poly[pollutant].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[pollutant].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 5. Simpan ke DataFrame final ----------
extracted_features_final = pd.DataFrame([combined_row])
print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = 'Dukun_gresik_polynomial.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

Jumlah fitur per polutan: 68
Total target fitur keseluruhan: 204

--- Memproses polutan: NO2 ---

--- Memproses polutan: SO2 ---



--- Memproses polutan: CO ---

Berhasil! Total kolom akhir yang dihasilkan: 204
File berhasil disimpan sebagai: Dukun_gresik_polynomial.csv
